# Cadence — local Whisper server for development

Runs `faster-whisper` on Colab's free GPU and exposes it over a public tunnel, so caption generation during development doesn't spend Groq's shared quota. See `colab/README.md` in the repo for the full picture — this notebook is just the server.

**Runtime**: make sure a GPU is attached before running — `Runtime` menu → `Change runtime type` → `T4 GPU`.

Run the three cells below in order. The last cell prints a URL — copy it into `COLAB_WHISPER_URL` in your `.env.local` and restart your dev server.

In [ ]:
# Cell 1 — install dependencies and the cloudflared tunnel binary.
!pip install -q fastapi uvicorn pydantic requests nest-asyncio faster-whisper
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# Cell 2 — load the model and define the server.
#
# "medium" with batched inference is the sweet spot: batching gives roughly a
# 3x speedup over sequential decoding at the same quality, which is what makes
# a three-hour episode a few minutes rather than a quarter of an hour. Move to
# "large-v3" for maximum quality, or "small" if you want it faster still.
#
# BATCH_SIZE trades VRAM for speed. 8 is comfortable on a free-tier T4; drop it
# to 4 if you hit out-of-memory, raise it on a bigger GPU.
#
# SHARED_SECRET is optional. Leave it blank to accept any request — the tunnel
# URL itself is an unguessable random string, which is a reasonable bar for a
# throwaway dev session. Set it here and mirror it in COLAB_SHARED_SECRET in
# .env.local for an actual check.

import os, tempfile, threading, uuid, traceback
import requests
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from faster_whisper import WhisperModel, BatchedInferencePipeline

MODEL_SIZE = "medium"
BATCH_SIZE = 8
SHARED_SECRET = ""

print(f"Loading {MODEL_SIZE} — this can take a minute or two the first time...")
_model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")
model = BatchedInferencePipeline(model=_model)
print("Model loaded.")

app = FastAPI()

# Jobs are held in memory and never persisted: a Colab runtime is ephemeral,
# and so is everything it was working on. The app treats an unknown job id the
# same as a dead server, which is exactly right after a restart.
JOBS: dict[str, dict] = {}

# One transcription on the GPU at a time — two concurrent runs on a free-tier
# T4 is a straightforward way to run out of VRAM. Jobs queue here instead.
_gpu = threading.Lock()


def _check_auth(authorization: str | None):
    if SHARED_SECRET and authorization != f"Bearer {SHARED_SECRET}":
        raise HTTPException(status_code=401, detail="Missing or wrong bearer token.")


@app.get("/health")
def health():
    # Deliberately unauthenticated: a health check that itself needs the
    # secret can't tell "down" apart from "wrong token" from the caller's
    # side, and the app's fallback logic only needs to know "is anything here".
    return {"ok": True, "model": MODEL_SIZE, "jobs": len(JOBS)}


class JobRequest(BaseModel):
    url: str


def _run_job(job_id: str, url: str):
    """Download the episode, transcribe it, and record the result."""
    path = None
    try:
        JOBS[job_id]["status"] = "downloading"
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            suffix = os.path.splitext(url.split("?")[0])[1][:5] or ".audio"
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
                for block in response.iter_content(chunk_size=1 << 20):
                    tmp.write(block)
                path = tmp.name
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"[{job_id[:8]}] downloaded {size_mb:.1f} MB, transcribing...")

        JOBS[job_id]["status"] = "transcribing"
        with _gpu:
            segments_gen, info = model.transcribe(
                path, batch_size=BATCH_SIZE, word_timestamps=True, vad_filter=True
            )

            segments, words, text_parts = [], [], []
            # faster-whisper yields lazily, so the real work happens in this
            # loop — which is why it stays inside the lock.
            for seg in segments_gen:
                text = seg.text.strip()
                if not text:
                    continue
                segments.append({"start": seg.start, "end": seg.end, "text": text})
                text_parts.append(text)
                for w in seg.words or []:
                    words.append({"word": w.word, "start": w.start, "end": w.end})

        JOBS[job_id].update(
            status="done",
            text=" ".join(text_parts),
            segments=segments,
            words=words,
        )
        print(
            f"[{job_id[:8]}] done — {len(segments)} segments "
            f"covering {info.duration / 60:.1f} min of audio."
        )
    except Exception as exc:
        traceback.print_exc()
        JOBS[job_id].update(status="error", error=f"{type(exc).__name__}: {exc}")
    finally:
        if path and os.path.exists(path):
            os.remove(path)


# The app hands over a URL rather than the audio itself, and polls this job id
# until it is done. That keeps every HTTP request short — which matters because
# the tunnel gives up on anything unanswered after ~100 seconds — and lets the
# episode be fetched once, here, straight from the podcast host over Google's
# network, instead of being pulled to a laptop and pushed back up.
@app.post("/jobs")
def create_job(body: JobRequest, authorization: str | None = Header(None)):
    _check_auth(authorization)
    job_id = str(uuid.uuid4())
    JOBS[job_id] = {"status": "queued"}
    threading.Thread(target=_run_job, args=(job_id, body.url), daemon=True).start()
    print(f"[{job_id[:8]}] queued")
    return {"job_id": job_id}


@app.get("/jobs/{job_id}")
def get_job(job_id: str, authorization: str | None = Header(None)):
    _check_auth(authorization)
    job = JOBS.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="Unknown job.")
    return job

In [ ]:
# Cell 3 — start the server and the tunnel, then print the public URL.
#
# Keep this cell running: stopping it (or the notebook idling out) ends the
# tunnel, and the app falls back to Groq automatically the next time it tries.

import nest_asyncio, uvicorn, subprocess, threading, time, re

nest_asyncio.apply()


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Waiting for the tunnel to come up...\n")
for line in proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print(f"Whisper server is live at: {url}")
        print(f"\nSet in .env.local:\n  COLAB_WHISPER_URL={url}")
        break